## **EDA - Detección de Problemas de Calidad**


In [0]:
%sql

-- VALORES INVALIDOS EN PRECIOS
WITH tickets AS (

  SELECT
    COUNT(*) AS ticket_total
  FROM ventas_EDA
),
problemas_calidad AS (

  SELECT
    COUNT(CASE WHEN precio IS NULL OR precio <= 0.01 THEN 1 END ) AS precio_invalido,
    COUNT(CASE WHEN cant IS NULL OR cant <= 0 THEN 1 END)  AS cant_invalido,
    COUNT(CASE WHEN total IS NULL OR total <= 0.01 THEN 1 END)  AS total_invalido
  FROM ventas_EDA
)

SELECT
  t.ticket_total,
  p.precio_invalido,
  ROUND(p.precio_invalido * 100.0 / t.ticket_total ,2) AS pct_precio_invalido,
  
  ROUND(p.cant_invalido * 100.0 / t.ticket_total,2) AS pct_cant_invalido,

  p.total_invalido,
  ROUND(p.total_invalido * 100.0 / t.ticket_total,2) AS pct_total_invalido
FROM tickets t, problemas_calidad p


In [0]:
%sql

-- REGISTROS DUPLICADOS
WITH duplicados AS (
  SELECT 
    venta, 
    articulo, 
    descrip,
    ROW_NUMBER() OVER (
      PARTITION BY venta, articulo, descrip
      ORDER BY articulo
    ) AS rn
  FROM ventas_EDA
)
SELECT * FROM duplicados WHERE rn > 1;

In [0]:
%sql
--MISMO ARTICULO CON DIFERENTAS DESCRIPCIONES
SELECT
  articulo,
  COUNT(DISTINCT TRIM(LOWER(descrip))) AS distintas_descripciones
FROM ventas_EDA
GROUP BY articulo 
  HAVING COUNT(DISTINCT TRIM(LOWER(descrip))) > 1
  ORDER BY distintas_descripciones DESC

In [0]:
%sql
-- OUTLIERS DE PRECIOS
WITH precios AS (
  SELECT precio
  FROM ventas_EDA
  WHERE vtaestado = 'normal'
    AND precio IS NOT NULL
    AND precio > 0
    AND precio <> 0.01
),

cuartiles AS (
  SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY precio) AS q1,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY precio) AS q3
  FROM precios
),

limites AS (
  SELECT
    q1,
    q3,
    (q3 - q1) AS iqr,
    (q1 - 1.5 * (q3 - q1)) AS limite_inferior,
    (q3 + 1.5 * (q3 - q1)) AS limite_superior
  FROM cuartiles
)

SELECT
  v.articulo,
  v.descrip,
  v.precio,
  l.limite_inferior,
  l.limite_superior,
  CASE
    WHEN v.precio < l.limite_inferior THEN 'OUTLIER_BAJO'
    WHEN v.precio > l.limite_superior THEN 'OUTLIER_ALTO'
  END AS tipo_outlier
FROM ventas_EDA v
CROSS JOIN limites l
WHERE v.vtaestado = 'normal'
  AND v.precio IS NOT NULL
  AND v.precio > 0
  AND v.precio <> 0.01
  AND (
    v.precio < l.limite_inferior
    OR v.precio > l.limite_superior
  )
ORDER BY v.precio DESC;